# Fashion MNIST Fast

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor

from common.paths import get_data_dir

In [ ]:
device = torch.device("mps") if torch.mps.is_available() else torch.device("cpu")

# Data

In [ ]:
dataset_split_seed = 123

In [ ]:
full_dataset = FashionMNIST(root=get_data_dir(), transform=ToTensor(), download=True)

train_dataset, val_dataset = random_split(
    full_dataset,
    lengths=[0.9, 0.1],
    generator=torch.random.manual_seed(dataset_split_seed),
)

test_dataset = FashionMNIST(
    root=get_data_dir(), transform=ToTensor(), download=True, train=False
)

In [ ]:
image, label = train_dataset[0]

plt.imshow(image[0].numpy())
plt.title(full_dataset.classes[label])

# Model

In [ ]:
from collections import OrderedDict
from collections.abc import Callable


@torch.compile
class SimpleLinearModel(nn.Module):
    def __init__(
        self,
        n_hidden_layers: int = 2,
        n_nodes_per_layer: int = 256,
        activation_fn: Callable = nn.ReLU,
    ) -> None:
        super().__init__()
        layers = OrderedDict()
        for layer_i in range(n_hidden_layers):
            layers[f"hidden{layer_i}"] = nn.LazyLinear(n_nodes_per_layer)
            layers[f"activ{layer_i}"] = activation_fn()
        layers["output"] = nn.LazyLinear(len(full_dataset.classes))
        self.network = nn.Sequential(layers)

    def forward(self, input: Tensor) -> Tensor:
        return self.network(input.flatten(1) if input.ndim > 2 else input.flatten())


simple_network = SimpleLinearModel()
simple_network

# Training Loop

In [ ]:
batch_size = 1_000  # Fraction of the dataset?  Nah...
learning_rate = 1e-3


train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
from dataclasses import dataclass


@dataclass
class Trainer:
    model: nn.Module
    train_dl: DataLoader[FashionMNIST]
    val_dl: DataLoader[FashionMNIST]
    optim: AdamW | None = None
    loss_func: nn.Module = nn.CrossEntropyLoss(reduction="sum")
    device = device

    def __post_init__(self) -> None:
        if self.optim is None:
            self.optim = AdamW(self.model.parameters(), lr=learning_rate)
        self.model = self.model.to(device)

    def train_epoch(self, epoch_idx: int) -> float:
        assert self.optim is not None
        self.model.train()
        for image_batch, label_batch in self.train_dl:
            self.optim.zero_grad()
            loss = self._model_loss(image_batch, label_batch)
            loss.backward()
            self.optim.step()
        return loss.item() / image_batch.shape[0]

    def val_epoch(self, epoch_idx: int) -> float:
        self.model.eval()
        with torch.inference_mode():
            loss = 0
            for image_batch, label_batch in self.val_dl:
                loss += self._model_loss(image_batch, label_batch)
        return loss.item() / len(self.val_dl.dataset)

    def _model_loss(self, image_data: Tensor, labels: Tensor) -> Tensor:
        image_data, labels = image_data.to(self.device), labels.to(self.device)
        estimate = self.model(image_data)
        encoded_labels = nn.functional.one_hot(
            labels, num_classes=len(self.train_dl.dataset.dataset.classes)
        ).to(torch.float32)
        return self.loss_func(estimate, encoded_labels)

    def loop(self, n_epoch: int) -> pd.DataFrame:
        output_data = pd.DataFrame(
            columns=["train_loss", "val_loss"],
            index=pd.Series(np.arange(n_epoch), name="epoch"),
        )
        for epoch_idx in range(n_epoch):
            output_data.at[epoch_idx, "train_loss"] = train_loss = self.train_epoch(
                epoch_idx
            )
            output_data.at[epoch_idx, "val_loss"] = val_loss = self.val_epoch(epoch_idx)
            print(f"{epoch_idx=}, {train_loss=}, {val_loss=}")
        return output_data

In [ ]:
trainer = Trainer(simple_network, train_dataloader, val_dataloader)
output_data = trainer.loop(40)
output_data.plot(grid=True, legend=True)

# Evaluation

In [ ]:
model = trainer.model

In [ ]:
eval_datas = []

for images, labels in val_dataloader:
    model_output = model(images.to(device))
    eval_data_single = pd.DataFrame(
        columns=["true_value", "model_top", "model_second"],
        index=pd.Series(np.arange(labels.shape[0]), name="val_idx"),
    )
    eval_data_single["true_value"] = labels.numpy()
    eval_data_single["model_top"] = np.argmax(
        model_output.to("cpu").detach().numpy(), axis=1
    )
    # eval_data_single["model_second"] = np.argmax(model_output.numpy(), axis=1)
    eval_datas.append(eval_data_single)
eval_data = pd.concat(eval_datas, axis="rows").reset_index(drop=True)
eval_data

In [ ]:
eval_data["matches"] = eval_data["true_value"] == eval_data["model_top"]
total_positive = eval_data.groupby("true_value")["matches"].count()
true_positive = eval_data.groupby("true_value")["matches"].sum()
model_positive = eval_data.groupby("model_top")["matches"].count()

confusion_df = pd.DataFrame(index=total_positive.index)
confusion_df["precision"] = true_positive / model_positive
confusion_df["recall"] = true_positive / total_positive
confusion_df

In [ ]:
a = Tensor([1, 2, 3])

In [ ]:
rows = 3
cols = 2

a = torch.ones(rows, cols, requires_grad=True)
a.register_hook(lambda x: print(f"a: {x}"))
b = torch.ones(rows, requires_grad=True)
b.register_hook(lambda x: print(f"b: {x}"))

dot = a @ (torch.rand(cols))
dot.register_hook(lambda x: print(f"dot: {x}"))
exp = torch.exp2(dot)
exp.register_hook(lambda x: print(f"exp: {x}"))
sum = exp + b
sum.register_hook(lambda x: print(f"sum: {x}"))
output = torch.sum(sum)

output.backward()

# print(a.grad)
# print(b.grad)